In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier  
from sklearn.preprocessing import LabelEncoder
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
BASE_ENV = Path().resolve().parent

In [ ]:
df = pd.read_csv(BASE_ENV / 'data/raw/cicids_wednesday_2018.csv')
df.head()

In [ ]:
columns_to_drop = []

In [ ]:
df = df.drop(columns='Timestamp')
columns_to_drop.append('Timestamp')

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
nan_counts = np.isnan(df.select_dtypes(include=[np.number])).sum()

print("Columns containing nan:")
print(nan_counts[nan_counts > 0])

In [ ]:
np.isinf(df.select_dtypes(include=[np.number])).sum()

In [ ]:
inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()

print("Columns containing nan:")
print(inf_counts[inf_counts > 0])

In [ ]:
# drop the columns with infinite and nan values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(axis=1)

In [ ]:
df.describe()

In [ ]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('/', '_')
df.columns

In [ ]:
df['Label'].value_counts()

In [ ]:
duplicate_names = df.columns[df.columns.str.contains(r'\.\d+$')]
print("Duplicate column names:")
print(duplicate_names.tolist())

columns_to_drop.extend(duplicate_names.tolist())
df = df.drop(columns=duplicate_names)

In [ ]:
hidden_clones = set()
for i in range(df.shape[1]):
    col_1 = df.iloc[:, i]
    
    for j in range(i + 1, df.shape[1]):
        col_2 = df.iloc[:, j]
        
        if col_1.equals(col_2):
            hidden_clones.add(df.columns[j])

print(f"clone columns: {hidden_clones}")
columns_to_drop.extend(list(hidden_clones))
df = df.drop(columns=hidden_clones)

In [ ]:
negative_values = (df.select_dtypes(include=[np.number]) < 0).sum()
print("Columns containing negative values:")
print(negative_values[negative_values > 0])

In [ ]:
df['Init_Fwd_Win_Byts'] = df['Init_Fwd_Win_Byts'].replace(-1, 0)
df['Init_Bwd_Win_Byts'] = df['Init_Bwd_Win_Byts'].replace(-1, 0)

negative_cols = df.select_dtypes(include=[np.number]).columns
df = df[(df[negative_cols] >= 0).all(axis=1)]

negative_values = (df.select_dtypes(include=[np.number]) < 0).sum()
print("Columns containing negative values:")
print(negative_values[negative_values > 0].tolist())

## Correlation testing 

In [ ]:
#df['Label'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)
encoder = LabelEncoder()
df['Label'] = encoder.fit_transform(df['Label'])


In [ ]:
df['Label'].value_counts()

In [ ]:
corr_matrix = df.corr().abs()
print("all Features Correlated with an Attack:")
target_corr = corr_matrix['Label'].sort_values(ascending=False)
print(target_corr.head(len(df.columns)))

In [ ]:
dead_correlation_columns = target_corr[target_corr < 0.02].to_dict()
print("Features with correlation < 0.02:")
print(dead_correlation_columns)
# total forward packets are highly correlated with labels (attacks) 
# but it is not showing in the correlation matrix 
# since the correlation matrix only captures linear relationships
# and the relationship between total forward packets and total backward packets may be non-linear.
# so we have to use other methods to capture the non-linear relationships between the features 
# and the target variable.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.barplot(x=list(dead_correlation_columns.values()), y=list(dead_correlation_columns.keys()), color='blue')
plt.title("Pearson Correlation Between Features and Labels", fontsize=14, pad=15, fontweight='bold')
plt.xlabel("Correlation Coefficient", fontsize=12)
plt.ylabel("Network Features", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
X = df.copy()
y = X['Label']
X = X.drop(columns=['Label'])

In [ ]:
rf_checker = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_checker.fit(X, y)

In [ ]:
important_features = pd.Series(rf_checker.feature_importances_, index=X.columns)
important_features = important_features.sort_values(ascending=False)

In [ ]:
dead_correlation_columns_RF = important_features[important_features <= 0.001].to_dict()
print("Features with low importance according to Random Forest:")
print(dead_correlation_columns_RF)

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.barplot(x=list(dead_correlation_columns_RF.values()), y=list(dead_correlation_columns_RF.keys()), color='blue')
plt.title("Features with low importance according to Random Forest", fontsize=14, pad=15, fontweight='bold')
plt.xlabel("importance vs all features", fontsize=12)
plt.ylabel("Network Features", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
columns_to_drop.extend(list(dead_correlation_columns_RF.keys()))
columns_to_drop